# 04 Evaluation Analysis

Inspect forecast outcomes, confidence calibration, and model-vs-baseline behavior. The API remains the system of record; notebooks generate analysis reports.

In [1]:
from pathlib import Path
import json

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sample = json.loads((REPO_ROOT / "data" / "samples" / "sample_analysis_export.json").read_text())
df = pd.DataFrame(sample["analyses"])
df

,analysis_id,ticker,horizon,article_count,included_article_count,aggregate_sentiment_score,agreement_score,evidence_strength_score,recent_momentum_score,volatility_score,predicted_percent_change,confidence_score,actual_percent_change,baseline_momentum_percent_change
0,sample-spy-001,SPY,3_trading_days,2,2,0.38,0.72,0.55,0.14,0.21,0.46,0.58,0.31,0.18
1,sample-aapl-001,AAPL,3_trading_days,1,1,-0.22,0.40,0.25,-0.08,0.33,-0.28,0.34,0.12,-0.10
2,sample-qqq-001,QQQ,3_trading_days,3,3,0.12,0.50,0.70,0.21,0.28,0.25,0.49,0.22,0.20


In [2]:
def direction(value: float, threshold: float = 0.05) -> str:
    if value > threshold:
        return "up"
    if value < -threshold:
        return "down"
    return "flat"

evaluated = df.copy()
evaluated["predicted_direction"] = evaluated["predicted_percent_change"].map(direction)
evaluated["actual_direction"] = evaluated["actual_percent_change"].map(direction)
evaluated["baseline_direction"] = evaluated["baseline_momentum_percent_change"].map(direction)
evaluated["direction_correct"] = evaluated["predicted_direction"] == evaluated["actual_direction"]
evaluated["baseline_direction_correct"] = evaluated["baseline_direction"] == evaluated["actual_direction"]
evaluated["absolute_error"] = (
    evaluated["predicted_percent_change"] - evaluated["actual_percent_change"]
).abs()
evaluated["baseline_absolute_error"] = (
    evaluated["baseline_momentum_percent_change"] - evaluated["actual_percent_change"]
).abs()
evaluated[["ticker", "predicted_direction", "actual_direction", "direction_correct", "absolute_error"]]

,ticker,predicted_direction,actual_direction,direction_correct,absolute_error
0,SPY,up,up,True,0.15
1,AAPL,down,up,False,0.40
2,QQQ,up,up,True,0.03


In [3]:
summary = pd.DataFrame(
    [
        {
            "forecasts": len(evaluated),
            "direction_accuracy": evaluated["direction_correct"].mean(),
            "baseline_direction_accuracy": evaluated["baseline_direction_correct"].mean(),
            "mean_absolute_error": evaluated["absolute_error"].mean(),
            "baseline_mean_absolute_error": evaluated["baseline_absolute_error"].mean(),
        }
    ]
)
summary

,forecasts,direction_accuracy,baseline_direction_accuracy,mean_absolute_error,baseline_mean_absolute_error
0,3,0.666667,0.666667,0.193333,0.123333


In [4]:
evaluated["confidence_bucket"] = pd.cut(
    evaluated["confidence_score"],
    bins=[0, 0.3, 0.6, 0.8, 1.0],
    labels=["low", "medium", "high", "very_high"],
    include_lowest=True,
)
calibration = evaluated.groupby("confidence_bucket", observed=True).agg(
    forecasts=("analysis_id", "count"),
    direction_accuracy=("direction_correct", "mean"),
    mean_absolute_error=("absolute_error", "mean"),
)
calibration

,forecasts,direction_accuracy,mean_absolute_error
confidence_bucket,,,
medium,3,0.666667,0.193333


In [5]:
report_path = REPO_ROOT / "data" / "reports" / "evaluation_summary_sample.csv"
report_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(report_path, index=False)
report_path

PosixPath('/Users/michaeldere/Workspace/aiml/micromarket/data/reports/evaluation_summary_sample.csv')